# Casual 장르 본격 분석용 게임 샘플링

## 목적
01 노트북에서 확인한 구간 경계(Q1/Q3)를 기반으로,
소형/중형/대형 구간별로 게임을 **랜덤 샘플링**하여 리뷰 수집 대상 목록을 확정한다.

## 01_casual_wilson_score.ipynb 와의 차이
- 01: Wilson Score 정렬 기반 극단값 추출 → 성공/실패 사례 탐색
- 02: 구간 내 랜덤 샘플링 → 본격 분석용 대표 게임 목록 확정

## 조건 (01과 동일)
- 장르: Casual
- 출시일: 2024-01-01 이후
- 최소 리뷰 수: 30건

In [1]:
import sys
sys.path.append('../../..')

import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv

from src.utils.db import get_connection

load_dotenv()

RANDOM_STATE = 42
MIN_REVIEWS  = 30
N_PER_TIER   = 10  # 구간별 샘플 수

In [2]:
conn = get_connection()
df = pd.read_sql(
    """
    SELECT appid, name_store, genres, positive, negative, release_date
    FROM steam_indie_list
    WHERE genres ILIKE '%Casual%'
    """,
    conn
)
conn.close()

df['total_reviews']      = df['positive'] + df['negative']
df['release_date_parsed'] = pd.to_datetime(df['release_date'], errors='coerce')
df = df[df['release_date_parsed'] >= '2024-01-01'].reset_index(drop=True)

print(f'Casual 장르 (2024-01-01 이후): {len(df):,}개')
df.head()

/var/folders/0b/g0grvv6j3wgdm_gjyyd_jb7m0000gn/T/ipykernel_50865/2298018442.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


Casual 장르 (2024-01-01 이후): 5,982개


,appid,name_store,genres,positive,negative,release_date,total_reviews,release_date_parsed
0,2379780,Balatro,"['Casual', 'Indie', 'Strategy']",150524,3042,"20 Feb, 2024",153566,2024-02-20
1,2709570,Supermarket Together,"['Casual', 'Indie', 'Simulation', 'Free To Play']",63994,3441,"9 Aug, 2024",67435,2024-08-09
2,3097560,Liar's Bar,"['Casual', 'Indie', 'Simulation', 'Strategy', ...",44341,4209,"2 Oct, 2024",48550,2024-10-02
3,2567870,Chained Together,"['Adventure', 'Casual', 'Indie', 'Simulation']",48180,4958,"19 Jun, 2024",53138,2024-06-19
4,2670630,Supermarket Simulator,"['Casual', 'Indie', 'Simulation']",64104,4237,"19 Jun, 2025",68341,2025-06-19


In [3]:
Z = 1.96

def wilson_lower_bound(positive: int, total: int, z: float = Z) -> float:
    if total < MIN_REVIEWS:
        return float('nan')
    p = positive / total
    z2 = z ** 2
    numerator = p + z2 / (2 * total) - z * np.sqrt(p * (1 - p) / total + z2 / (4 * total ** 2))
    return numerator / (1 + z2 / total)

df['wilson_score'] = df.apply(
    lambda r: wilson_lower_bound(r['positive'], r['total_reviews']), axis=1
)
df_valid = df.dropna(subset=['wilson_score']).copy()

# 01 노트북과 동일한 사분위수 기준으로 구간 설정
q1 = df_valid['total_reviews'].quantile(0.25)
q3 = df_valid['total_reviews'].quantile(0.75)

def assign_size_tier(total: int) -> str:
    if total <= q1:
        return '소형'
    elif total <= q3:
        return '중형'
    return '대형'

df_valid['size_tier'] = df_valid['total_reviews'].apply(assign_size_tier)

print(f'Wilson Score 계산 가능 게임: {len(df_valid):,}개')
print(f'\n구간 경계: 소형 {MIN_REVIEWS}~{int(q1)}건 / 중형 {int(q1)+1}~{int(q3)}건 / 대형 {int(q3)+1}건+')
print(f'\n구간별 게임 수:')
print(df_valid['size_tier'].value_counts()[['소형', '중형', '대형']].to_string())

Wilson Score 계산 가능 게임: 1,714개

구간 경계: 소형 30~48건 / 중형 49~294건 / 대형 295건+

구간별 게임 수:
size_tier
소형    430
중형    856
대형    428


### 구간 설정

01 노트북과 동일한 Q1/Q3 경계를 사용한다.
각 구간 내 게임 수가 `N_PER_TIER`보다 충분히 많아야 랜덤 샘플링이 의미 있다.

In [4]:
tiers = ['소형', '중형', '대형']
samples = []

for tier in tiers:
    group = df_valid[df_valid['size_tier'] == tier]
    n = min(N_PER_TIER, len(group))
    sampled = group.sample(n=n, random_state=RANDOM_STATE).copy()
    sampled['size_tier'] = tier
    samples.append(sampled)
    print(f'{tier}: 모집단 {len(group):,}개 중 {n}개 샘플링')

df_sample = pd.concat(samples).reset_index(drop=True)

print(f'\n총 샘플: {len(df_sample)}개')
cols = ['size_tier', 'name_store', 'total_reviews', 'positive', 'negative', 'wilson_score']
print(df_sample[cols].sort_values(['size_tier', 'wilson_score'], ascending=[True, False]).to_string(index=False))

소형: 모집단 430개 중 10개 샘플링
중형: 모집단 856개 중 10개 샘플링
대형: 모집단 428개 중 10개 샘플링

총 샘플: 30개
size_tier                                    name_store  total_reviews  positive  negative  wilson_score
       대형                            shapez 2 - Factory          11061     10856       205      0.978781
       대형                Find 100 Ducks and Blast Them!            333       324         9      0.949440
       대형                                      Spilled!           2592      2470       122      0.944087
       대형                                  Ball-it Hell            321       311        10      0.943613
       대형                         Dusk Pub - Adult Only           1820      1690       130      0.915815
       대형                           Trials of Innocence            758       708        50      0.914087
       대형                           Fling to the Finish           1256      1086       170      0.844614
       대형                     TRADESMAN: Deal to Dealer           1160       959

### 랜덤 샘플링 결과

`RANDOM_STATE=42`로 재현 가능한 샘플을 추출했다.
각 구간에서 Wilson Score 분포를 확인하여 샘플이 구간 전체를 고르게 대표하는지 검증한다.

In [5]:
# 샘플 Wilson Score 분포가 모집단을 잘 대표하는지 확인
print('=== 구간별 Wilson Score: 모집단 vs 샘플 비교 ===')
for tier in tiers:
    pop   = df_valid[df_valid['size_tier'] == tier]['wilson_score']
    samp  = df_sample[df_sample['size_tier'] == tier]['wilson_score']
    print(f'\n[{tier}]')
    print(f'  모집단  mean={pop.mean():.3f}  median={pop.median():.3f}  std={pop.std():.3f}')
    print(f'  샘플    mean={samp.mean():.3f}  median={samp.median():.3f}  std={samp.std():.3f}')

# 시각화
fig = px.box(
    df_valid,
    x='size_tier',
    y='wilson_score',
    color='size_tier',
    category_orders={'size_tier': tiers},
    title='Casual 장르 — 구간별 Wilson Score 분포 (모집단)',
    labels={'size_tier': '규모 구간', 'wilson_score': 'Wilson Score'},
    points=False,
    height=420,
)

# 샘플 포인트 오버레이
for i, tier in enumerate(tiers):
    samp = df_sample[df_sample['size_tier'] == tier]
    fig.add_scatter(
        x=[tier] * len(samp),
        y=samp['wilson_score'],
        mode='markers',
        marker=dict(color='black', size=8, symbol='diamond'),
        name='샘플' if i == 0 else None,
        showlegend=(i == 0),
    )

fig.update_layout(showlegend=True)
fig.show()

=== 구간별 Wilson Score: 모집단 vs 샘플 비교 ===

[소형]
  모집단  mean=0.710  median=0.758  std=0.176
  샘플    mean=0.721  median=0.753  std=0.127

[중형]
  모집단  mean=0.775  median=0.811  std=0.143
  샘플    mean=0.831  median=0.836  std=0.074

[대형]
  모집단  mean=0.837  median=0.867  std=0.121
  샘플    mean=0.885  median=0.915  std=0.077


### 샘플 대표성 검증

샘플의 평균·중위값이 모집단과 크게 차이 나지 않으면 대표성이 확보된 것이다.
박스플롯에 오버레이된 샘플 포인트(◆)가 박스 범위 내에 고르게 분포하는지 확인한다.

편향이 의심되면 `RANDOM_STATE` 값을 바꿔 재샘플링하거나 `N_PER_TIER`를 늘린다.

In [6]:
output_path = '../../../data/processed/casual_analysis_sample.csv'
df_sample[['appid', 'name_store', 'size_tier', 'positive', 'negative', 'total_reviews', 'wilson_score', 'release_date']].to_csv(
    output_path, index=False
)
print(f'저장 완료: {output_path}')
print(f'총 {len(df_sample)}개 게임 (구간별 {N_PER_TIER}개 × 3구간)')

저장 완료: ../../../data/processed/casual_analysis_sample.csv
총 30개 게임 (구간별 10개 × 3구간)
